# Bangladesh: 0224_climate_bootcamp_session8

Static adaptation only; Earth Engine/data/asset execution not verified.

Original scientific assumptions and date windows retained; use the national workflow for QA improvements.

Review scale, missing data, units, and any training/calibration requirements before interpreting results.

Experimental model: use spatial/time holdout validation and independent Bangladesh ground truth; finer pixels do not prove added information.

May contain tutorial-specific assets or imported training features; inspect dependencies before execution.

In [ ]:
# Bangladesh adaptation: 0224_climate_bootcamp_session8
# Original author: Amirhossein Ahrari; see original attribution below and originals/.
from bdgee.core import initialize, boundaries
from bdgee.notebook import export_outputs, upload_local
initialize()  # Run `earthengine authenticate` first; project is configured in bdgee.
BD_COUNTRY, BD_DISTRICTS = boundaries()
BD_ROI = BD_COUNTRY.geometry()
BD_BBOX = [88.0, 20.5, 92.7, 26.7]  # search only, not analysis boundary
roi = geometry = border = iran = tehran = province = loc = point = BD_ROI
# Run from the repository root; all local outputs are isolated by source.
from pathlib import Path
import os
_REPO_ROOT = Path.cwd()
_OUTPUT_DIR = Path(os.environ.get('BDGEE_OUTPUT_DIR', _REPO_ROOT / 'outputs' / 'climate' / '0224_climate_bootcamp_session8'))
_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(_OUTPUT_DIR)


In [ ]:
"""08_class8_pr_downscaling.ipynb

Tutorial Code by Amirhossein Ahrari
YouTube: https://www.youtube.com/@amirhosseinahrarigee
Tutorial Video: Climate Data Analysis Bootcamp using GEE and Python - Session8: ERA5 Precipitation Downscaling
This code is part of a tutorial series on Earth Engine programming techniques
presented on the Amirhossein Ahrari YouTube channel. You are free to use and modify
this code for academic and non-academic purposes. Don't forget to subscribe to
the Amirhossein Ahrari channel and follow the videos to support the instructor

"""
import ee
None
None
'# **Study Area**'
import geemap
map = geemap.Map(basemap='SATELLITE')
map
roi = BD_ROI
roi
'# **Image Collection**'
era5 = ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR').filterDate('2015', '2026').select(['total_precipitation_sum'], ['pr'])
era5
time_start = ee.Date('2015')
time_end = ee.Date('2026')
time_dif = time_end.difference(time_start, 'month').round()
time_list = ee.List.sequence(0, ee.Number(time_dif).subtract(1)).map(lambda x: time_start.advance(x, 'month'))
time_list

def monthly(date_list, col):
    start_month = ee.Date(date_list)
    end_month = start_month.advance(1, 'month')
    col_month = col.filterDate(start_month, end_month).mean()
    return col_month.set('system:time_start', start_month.millis())
ndvi = ee.ImageCollection('MODIS/061/MOD13A2').filterDate(time_start, time_end).select('NDVI', 'EVI')
lst = ee.ImageCollection('MODIS/061/MOD11A2').filterDate(time_start, time_end).select('LST_Day_1km', 'LST_Night_1km')
et = ee.ImageCollection('MODIS/061/MOD16A2GF').filterDate(time_start, time_end).select('ET', 'PET')
snow = ee.ImageCollection('MODIS/061/MOD10A2').filterDate(time_start, time_end).select('Maximum_Snow_Extent')
topo = ee.ImageCollection('JAXA/ALOS/AW3D30/V4_1').mean().select('DSM')
lc = ee.ImageCollection('MODIS/061/MCD12Q1').filterDate(time_start, time_end).select('LC_Type1').mode()
ndvi_monthly = ee.ImageCollection(time_list.map(lambda x: monthly(x, ndvi)))
lst_monthly = ee.ImageCollection(time_list.map(lambda x: monthly(x, lst)))
et_monthly = ee.ImageCollection(time_list.map(lambda x: monthly(x, et)))
snow_monthly = ee.ImageCollection(time_list.map(lambda x: monthly(x, snow)))
monthly_col = ndvi_monthly.combine(lst_monthly).combine(et_monthly).combine(snow_monthly).map(lambda img: img.addBands(topo).addBands(lc))
monthly_col
'# **to Xarray**'
from xee import helpers
from shapely.geometry import shape
import xarray as xr
grid_10km = helpers.fit_geometry(geometry=shape(roi.getInfo()), grid_crs='EPSG:4326', grid_scale=(0.1, -0.1))
ds_10km = xr.open_dataset(monthly_col, engine='ee', **grid_10km)
ds_10km = ds_10km.sortby('time') * 1
pr_10km = xr.open_dataset(era5, engine='ee', **grid_10km)
pr_10km = pr_10km.sortby('time') * 1000
ds_10km = xr.merge([ds_10km, pr_10km])
ds_10km['lon'] = ds_10km.x.broadcast_like(ds_10km.NDVI)
ds_10km['lat'] = ds_10km.y.broadcast_like(ds_10km.NDVI)
ds_10km['month'] = ds_10km.time.dt.month.broadcast_like(ds_10km.NDVI)
'# **Sampling**'
df_10km = ds_10km.to_dataframe()
df_10km
x = df_10km.drop('pr', axis=1)
x
y = df_10km['pr']
y
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
'# **Modelling**'
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(x_train, y_train)
from sklearn.metrics import mean_squared_error, r2_score
y_pred = model.predict(x_test)
import numpy as np
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
rmse
r2 = r2_score(y_test, y_pred)
r2
'# **Prediction**'
grid_1km = helpers.fit_geometry(geometry=shape(roi.getInfo()), grid_crs='EPSG:4326', grid_scale=(0.01, -0.01))
ds_1km = xr.open_dataset(monthly_col, engine='ee', **grid_1km)
ds_1km = ds_1km.sortby('time') * 1
ds_1km['lon'] = ds_1km.x.broadcast_like(ds_1km.NDVI)
ds_1km['lat'] = ds_1km.y.broadcast_like(ds_1km.NDVI)
ds_1km['month'] = ds_1km.time.dt.month.broadcast_like(ds_1km.NDVI)
df_1km = ds_1km.to_dataframe()
df_1km
df_1km['pr_pred'] = model.predict(df_1km)
pr_1km = df_1km['pr_pred'].to_xarray()
pr_10km.pr.isel(time=100).plot(x='x', y='y', robust=True)
pr_1km.isel(time=100).plot(x='x', y='y', robust=True)


In [ ]:

# Review these task plans; task creation does not start exports.
os.chdir(_REPO_ROOT)
bd_tasks = export_outputs(globals(), 'climate', '0224_climate_bootcamp_session8', scale=1000)
# Set an appropriate native scale above before creating plans.
# for task in bd_tasks: task.start()
# upload_local(_OUTPUT_DIR, 'climate')  # explicit upload; requires Drive OAuth scope
